# 🚀 LSTM Product Training - SHORT

**Configuración:** 30 días → 7 días (predicción semanal)

**Plataforma:** Kaggle (GPU T4 x2) o Colab Pro (A100)

**Tiempo estimado:**
- Colab Pro A100: ~45-60 min ⚡
- Colab Pro L4: ~1.5-2h
- Kaggle T4 x2: ~2-3h

In [ ]:
# Verificar GPU
import tensorflow as tf
import gc

gc.collect()

print("="*80)
print("PRODUCT SHORT: 30→7 días")
print("="*80)
print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"\nGPUs detectadas: {len(gpus)}")

if len(gpus) > 0:
    for i, gpu in enumerate(gpus):
        print(f"   GPU {i}: {gpu.name}")
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"\n✅ Usando GPU para entrenamiento")
else:
    print("\n⚠️ NO GPU DETECTADA")

print("="*80)

In [ ]:
# Instalar dependencias
!pip install -q openpyxl seaborn
print("\n✅ Dependencias instaladas")
gc.collect()

In [ ]:
# Detectar plataforma y configurar rutas
import os
import sys
import shutil

IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules

if IS_KAGGLE:
    print("📍 Plataforma: KAGGLE")
    BASE_PATH = '/kaggle/working'
    datasets = [d for d in os.listdir('/kaggle/input/') if d != 'sample_submission']
    if datasets:
        DATASET_DIR = datasets[0]
        DATA_PATH = f'/kaggle/input/{DATASET_DIR}/online_retail_2.xlsx'
        SCRIPT_PATH = f'/kaggle/input/{DATASET_DIR}/train_products_temporal.py'
        print(f"   Dataset: {DATASET_DIR}")
    
    os.makedirs(f'{BASE_PATH}/data/processed', exist_ok=True)
    os.makedirs(f'{BASE_PATH}/models/temporal/products/short', exist_ok=True)
    
    shutil.copy2(DATA_PATH, f'{BASE_PATH}/data/processed/online_retail_2.xlsx')
    shutil.copy2(SCRIPT_PATH, f'{BASE_PATH}/train_products_temporal.py')
    
elif IS_COLAB:
    print("📍 Plataforma: COLAB")
    BASE_PATH = '/content'
    from google.colab import files
    print("\n📤 Sube 'online_retail_2.xlsx'")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]
    print("\n📤 Sube 'train_products_temporal.py'")
    uploaded = files.upload()
    SCRIPT_PATH = list(uploaded.keys())[0]
    
    os.makedirs(f'{BASE_PATH}/data/processed', exist_ok=True)
    os.makedirs(f'{BASE_PATH}/models/temporal/products/short', exist_ok=True)
    
    shutil.move(DATA_PATH, f'{BASE_PATH}/data/processed/online_retail_2.xlsx')
    shutil.move(SCRIPT_PATH, f'{BASE_PATH}/train_products_temporal.py')

print(f"\n✅ Archivos listos en {BASE_PATH}")
gc.collect()

In [ ]:
# Importar script
sys.path.append(BASE_PATH)
from train_products_temporal import ProductTemporalAnalyzer, ProductTemporalConfig

print("✅ Script importado")
print(f"📊 Config SHORT: {ProductTemporalConfig.SHORT['window_days']}→{ProductTemporalConfig.SHORT['forecast_days']}d")
gc.collect()

In [ ]:
# Preparar datos
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

start_time = datetime.now()
print(f"⏰ Inicio: {start_time}\n")

analyzer = ProductTemporalAnalyzer(
    data_path=f'{BASE_PATH}/data/processed/online_retail_2.xlsx',
    output_dir=f'{BASE_PATH}/models/temporal/products'
)

analyzer.load_and_preprocess_data()
gc.collect()

top_n_products = 50
analyzer.select_products_by_volume(top_n=top_n_products, min_transactions=20)
gc.collect()

print(f"\n✅ {len(analyzer.products)} productos listos")

In [ ]:
# ENTRENAR SHORT
import time

print("="*70)
print("ENTRENAMIENTO SHORT (30→7 días)")
print("="*70)

t0 = time.time()
results = analyzer.train_all_products(ProductTemporalConfig.SHORT)
mins = (time.time() - t0) / 60

print(f"\n✅ COMPLETADO ({mins:.1f} min)")
print(f"Productos entrenados: {results['trained_count']}")
print(f"MAE promedio: {results['avg_mae']:.2f}")

with open(f'{BASE_PATH}/models/temporal/products/short/RESULTADOS.txt', 'w') as f:
    f.write(f"SHORT - RESULTADOS\n")
    f.write(f"Productos: {results['trained_count']}\n")
    f.write(f"Tiempo: {mins:.1f} min\n")
    f.write(f"MAE: {results['avg_mae']:.2f}\n")

total = (datetime.now() - start_time).total_seconds() / 60
print(f"\n⏰ Tiempo total: {total:.1f} min")

In [ ]:
# Comprimir modelos
!cd {BASE_PATH}/models/temporal && zip -r products_short.zip products/short/
print("✅ Modelos comprimidos: products_short.zip")
!ls -lh {BASE_PATH}/models/temporal/*.zip